# Lab AWS — Titanic: SageMaker Training Job (XGBoost Built-in)

## 1. Preparación del entorno

In [ ]:
import boto3
import numpy as np
import pandas as pd
import sagemaker
from sagemaker.inputs import TrainingInput
from sagemaker.serializers import CSVSerializer
from IPython.display import display

sess   = sagemaker.Session()
role   = boto3.client('iam').get_role(RoleName='LabRole')['Role']['Arn']
bucket = sess.default_bucket()
prefix = "sagemaker/titanic"

print(f"Bucket : s3://{bucket}")
print(f"Region : {sess.boto_region_name}")
print(f"Role   : {role}")

## 2. Obtener la URI del contenedor XGBoost

In [ ]:
container = sagemaker.image_uris.retrieve(
    "xgboost",
    sess.boto_region_name,
    "1.7-1"
)
print(f"Contenedor XGBoost: {container}")

## 3. Canales de datos (outputs del Processing Job)

In [ ]:
s3_input_train = TrainingInput(
    s3_data=f"s3://{bucket}/{prefix}/train",
    content_type="csv"
)
s3_input_validation = TrainingInput(
    s3_data=f"s3://{bucket}/{prefix}/validation",
    content_type="csv"
)

print(f"train      : s3://{bucket}/{prefix}/train")
print(f"validation : s3://{bucket}/{prefix}/validation")

## 4. Configurar y lanzar el Estimator XGBoost

Tarda entre **5 y 10 minutos**.

In [ ]:
xgb = sagemaker.estimator.Estimator(
    container,
    role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/model-output",
    base_job_name="titanic-xgboost",
    sagemaker_session=sess,
)

xgb.set_hyperparameters(
    objective="binary:logistic",
    num_round=150,
    max_depth=5,
    eta=0.1,
    gamma=4,
    min_child_weight=6,
    subsample=0.8,
    eval_metric="auc",
    verbosity=1,
)

xgb.fit(
    {"train": s3_input_train, "validation": s3_input_validation},
    wait=True,
)

print("Training job finalizado.")

## 5. Desplegar endpoint y evaluar

In [ ]:
predictor = xgb.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    serializer=CSVSerializer()
)
print(f"Endpoint activo: {predictor.endpoint_name}")

In [ ]:
s3 = boto3.client("s3")
s3.download_file(bucket, f"{prefix}/test/test.csv", "test.csv")

test_df = pd.read_csv("test.csv", header=None)
X_test  = test_df.iloc[:, 1:]
y_test  = test_df.iloc[:, 0]
print(f"Test set: {test_df.shape}")

In [ ]:
def predict_batches(predictor, data, batch_size=100):
    batches = np.array_split(data, max(1, len(data) // batch_size))
    raw = ""
    for batch in batches:
        raw += predictor.predict(batch).decode("utf-8")
    return np.array([float(x) for x in raw.strip().split("\n") if x])

probs = predict_batches(predictor, X_test.to_numpy())
preds = (probs >= 0.5).astype(int)
print(f"Primeras predicciones: {preds[:10]}")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
print()
print(classification_report(y_test, preds, target_names=["No sobrevivio", "Sobrevivio"]))

cm = pd.crosstab(y_test, pd.Series(preds, name="Pred"), rownames=["Real"])
display(cm)

## 6. Limpieza — eliminar endpoint para evitar cargos

In [ ]:
predictor.delete_endpoint()
print("Endpoint eliminado.")